# SELF-RAG Pipeline
### PDF → Chunking → FAISS → Retrieve? → Relevance → Generate → Grounded? → Useful? → Answer

In [ ]:
!pip install langchain langchain-community langchain-ollama langchain-text-splitters faiss-cpu pypdf -q

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings, ChatOllama

c:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Load PDF

In [2]:
loader = PyPDFLoader("A._P._J._Abdul_Kalam.pdf")
pages = loader.load()

print(f"Total Pages: {len(pages)}")
print(f"Sample Content:\n{pages[0].page_content[:500]}")

Total Pages: 30
Sample Content:
A. P. J. Abdul Kalam
Official portrait, c. 2002
President of India
In office
25 July 2002 – 25 July 2007
Prime Minister Atal Bihari Vajpayee
Manmohan Singh
Vice President Krishan Kant
Bhairon Singh Shekhawat
Preceded by K. R. Narayanan
Succeeded by Pratibha Patil
Principal Scientific Adviser to the
Government of India
In office
November 1999 – November 2001
President K. R. Narayanan
Prime Minister Atal Bihari Vajpayee
Preceded by Office established
Succeeded by Rajagopala Chidambaram
Director Ge


## Step 2: Split into Chunks

In [3]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(pages)

print(f"Total Chunks: {len(chunks)}")
print(f"Sample Chunk:\n{chunks[0].page_content}")

Total Chunks: 136
Sample Chunk:
A. P. J. Abdul Kalam
Official portrait, c. 2002
President of India
In office
25 July 2002 – 25 July 2007
Prime Minister Atal Bihari Vajpayee
Manmohan Singh
Vice President Krishan Kant
Bhairon Singh Shekhawat
Preceded by K. R. Narayanan
Succeeded by Pratibha Patil
Principal Scientific Adviser to the
Government of India
In office
November 1999 – November 2001
President K. R. Narayanan
Prime Minister Atal Bihari Vajpayee
Preceded by Office established
Succeeded by Rajagopala Chidambaram
Director General of Defence Research and
Development Organisation
In office
1992–1999
A. P. J. Abdul Kalam
Avul Pakir Jainulabdeen Abdul Kalam (/ˈʌbdʊl
kəˈlɑːm/ ⓘ UB-duul k ə -LAHM; 15 October 1931 –
27 July 2015) was an Indian aerospace scientist and
statesman who served as the president of India from
2002 to 2007.
Born and raised in a Muslim family in Rameswaram,
Tamil Nadu, Kalam studied physics and aerospace
engineering. He spent the next four decades as a


## Step 3: Create Embeddings & Store in FAISS

In [4]:
embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")

vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

print(f"Vector Store Created with {vector_store.index.ntotal} vectors")

Vector Store Created with 136 vectors


## Step 4: Setup LLM

In [5]:
llm = ChatOllama(model="llama3.2:3b", temperature=0)

## Step 5: RETRIEVE Token — Does the query need retrieval?

In [6]:
def check_retrieve(query):
    prompt = f"""Decide if retrieval from a knowledge base is needed to answer this query.
If the query asks about specific facts, people, events, or details → say "yes".
If it is a general greeting or simple math → say "no".

Query: {query}
Answer with only "yes" or "no"."""

    response = llm.invoke(prompt).content.strip().lower()
    return "yes" in response

## Step 6: RELEVANCE Token — Filter irrelevant chunks

In [7]:
def filter_relevant(query, docs):
    relevant = []
    for doc in docs:
        prompt = f"""Is the following chunk relevant to answering the query?

Query: {query}
Chunk: {doc.page_content[:500]}

Answer with only "relevant" or "irrelevant"."""

        response = llm.invoke(prompt).content.strip().lower()
        if "relevant" in response and "irrelevant" not in response:
            relevant.append(doc)
    return relevant

## Step 7: Generate Answer from Filtered Context

In [8]:
def generate_answer(query, docs):
    context = "\n\n".join([doc.page_content for doc in docs])
    prompt = f"""Answer the question based only on the following context:

{context}

Question: {query}
Answer:"""
    return llm.invoke(prompt).content

## Step 8: SUPPORT Token — Is the answer grounded in context?

In [9]:
def check_support(answer, docs):
    context = "\n\n".join([doc.page_content[:500] for doc in docs])
    prompt = f"""Is the following answer fully supported by the context? Check if every claim in the answer can be found in the context.

Context: {context}

Answer: {answer}

Respond with only "fully supported", "partially supported", or "not supported"."""

    response = llm.invoke(prompt).content.strip().lower()
    return response

## Step 9: UTILITY Token — Is the answer useful and complete?

In [10]:
def check_utility(query, answer):
    prompt = f"""Rate how well the answer addresses the question.

Question: {query}
Answer: {answer}

Respond with only a score from 1 to 5 where:
1 = not useful at all
5 = perfectly answers the question"""

    response = llm.invoke(prompt).content.strip()
    try:
        score = int(''.join(c for c in response if c.isdigit())[:1])
    except (ValueError, IndexError):
        score = 3
    return score

## Step 10: Full Self-RAG Pipeline

In [11]:
def self_rag(query):
    print(f"Query: {query}\n")

    # RETRIEVE token
    needs_retrieval = check_retrieve(query)
    print(f"[RETRIEVE] Needs retrieval: {needs_retrieval}")

    if not needs_retrieval:
        print("[DIRECT LLM] Answering without retrieval...")
        answer = llm.invoke(query).content
        print(f"\nAnswer: {answer}")
        return answer

    # Retrieve candidate chunks
    retrieved_docs = retriever.invoke(query)
    print(f"[RETRIEVE] Got {len(retrieved_docs)} candidate chunks")

    # RELEVANCE token
    relevant_docs = filter_relevant(query, retrieved_docs)
    print(f"[RELEVANCE] {len(relevant_docs)}/{len(retrieved_docs)} chunks are relevant")

    if not relevant_docs:
        print("[FALLBACK] No relevant chunks found, using all retrieved chunks...")
        relevant_docs = retrieved_docs

    # Generate answer
    answer = generate_answer(query, relevant_docs)
    print(f"[GENERATE] Answer: {answer}")

    # SUPPORT token
    support = check_support(answer, relevant_docs)
    print(f"[SUPPORT] Grounded: {support}")

    # UTILITY token
    utility = check_utility(query, answer)
    print(f"[UTILITY] Score: {utility}/5")

    # Re-retrieve if not useful enough
    if utility < 3:
        print("\n[RE-RETRIEVE] Low utility, fetching more chunks...")
        more_docs = vector_store.as_retriever(search_kwargs={"k": 5}).invoke(query)
        relevant_docs = filter_relevant(query, more_docs)
        if relevant_docs:
            answer = generate_answer(query, relevant_docs)
            print(f"[RE-GENERATE] Answer: {answer}")

    print(f"\nFinal Answer: {answer}")
    return answer

## Run Self-RAG

In [12]:
answer = self_rag("Where was Abdul Kalam born?")

Query: Where was Abdul Kalam born?

[RETRIEVE] Needs retrieval: True
[RETRIEVE] Got 3 candidate chunks
[RELEVANCE] 2/3 chunks are relevant
[GENERATE] Answer: Abdul Kalam was born in Rameswaram, Tamil Nadu.
[SUPPORT] Grounded: partially supported.
[UTILITY] Score: 2/5

[RE-RETRIEVE] Low utility, fetching more chunks...
[RE-GENERATE] Answer: According to the text, Abdul Kalam was born in Rameswaram, Tamil Nadu.

Final Answer: According to the text, Abdul Kalam was born in Rameswaram, Tamil Nadu.


## Test: Query that doesn't need retrieval

In [13]:
answer = self_rag("What is 2 + 2?")

Query: What is 2 + 2?

[RETRIEVE] Needs retrieval: False
[DIRECT LLM] Answering without retrieval...

Answer: 2 + 2 = 4


Test Questions

1. What year was Abdul Kalam born?
2. What was the name of the coronary stent Kalam co-developed?
3. Which institution did Kalam study aerospace engineering at?
4. How many electoral votes did Kalam receive in the 2002 presidential election?
5. What was Kalam's role at DRDO from 1992 to 1999?